# Create dataset


In [1]:
import math
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import torch
from rtmlib import Wholebody

In [2]:
COCO_KPT = {
    'nose': 0,
    'left_eye': 1,
    'right_eye': 2,
    'left_shoulder': 5,
    'right_shoulder': 6,
    'left_elbow': 7,
    'right_elbow': 8,
    'left_wrist': 9,
    'right_wrist': 10,
    'left_hip': 11,
    'right_hip': 12,
}


def is_facing_camera(kpts, conf, thr=0.5, sym_thr=0.2):
    # require nose, eyes, shoulders
    for part in ('nose', 'left_eye', 'right_eye', 'left_shoulder', 'right_shoulder'):
        if conf[COCO_KPT[part]] < thr:
            return False
    # shoulders roughly level
    yL = kpts[COCO_KPT['left_shoulder'], 1]
    yR = kpts[COCO_KPT['right_shoulder'], 1]
    if abs(yL - yR) > sym_thr * (yL + yR):
        return False
    return True


def arms_visible(conf, thr=0.5):
    for part in ('left_elbow', 'right_elbow', 'left_wrist', 'right_wrist'):
        if conf[COCO_KPT[part]] < thr:
            return False
    return True


def waist_up(conf, thr=0.5):
    return conf[COCO_KPT['left_hip']] >= thr and conf[COCO_KPT['right_hip']] >= thr


def check_interpreter(wholebody, image):
    keypoints, scores = wholebody(image)

    if keypoints.shape[0] != 1:
        return False

    return (
        is_facing_camera(keypoints[0], scores[0])
        and arms_visible(scores[0])
        and waist_up(scores[0])
    )

In [3]:
width, height = 1280, 720
fps = 25


def pad_frame(frame, target_w=250, target_h=219):
    h, w = frame.shape[:2]

    pad_width = target_w - w
    pad_left = pad_width // 2
    pad_right = pad_width - pad_left

    pad_height = target_h - h
    pad_top = pad_height // 2
    pad_bottom = pad_height - pad_top

    padding_config = ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0))
    frame = np.pad(frame, padding_config, mode='constant', constant_values=0)

    return frame


def crop_frame(frame, path):
    split = path.split(os.sep)
    if len(split) < 4:
        raise ValueError()

    year = int(split[-4])
    month = int(split[-3])

    if year > 2023 or (year == 2023 and month == 12):
        x_start, x_end = int(0.755 * width), int(0.950 * width)
        y_start, y_end = int(0.400 * height), int(0.705 * height)
    else:
        x_start, x_end = int(0.800 * width), int(0.945 * width)
        y_start, y_end = int(0.660 * height), int(0.945 * height)

    return frame[y_start:y_end, x_start:x_end]


def extract_frames(wholebody, video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None, None

    target_frames = 40

    step = math.ceil(total_frames / target_frames)

    with_interpreter = []
    without_interpreter = []

    idx = 0
    while idx < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            break

        frame = crop_frame(frame, video_path)

        if check_interpreter(wholebody, frame):
            with_interpreter.append(pad_frame(frame))
        else:
            without_interpreter.append(pad_frame(frame))

        idx += step

    cap.release()

    return with_interpreter, without_interpreter


dirs = [2025, 2024, 2023, 2022, 2021, 2020, 2019]
video_paths = []
for base_dir in dirs:
    for root, _, files in os.walk(str(base_dir)):
        if '.has_interpreter' in files:
            video_path = os.path.join(root, 'video.mp4')
            video_paths.append(video_path)

print(f'[main] {len(video_paths)} videos to process')


def init_worker():
    global wholebody
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    wholebody = Wholebody(mode='lightweight', backend='onnxruntime', device=device)
    print(f'[tid={threading.get_ident()}] wholebody loaded on {device}')


def process_video_wrapper(video_path):
    try:
        with_interp_frames, without_interp_frames = extract_frames(
            wholebody, video_path
        )

        if with_interp_frames is not None and without_interp_frames is not None:
            output_dir = os.path.dirname(video_path)

            if with_interp_frames:
                with_interp_array = np.array(with_interp_frames)
                np.save(
                    os.path.join(output_dir, 'with_interpreter_frames.npy'),
                    with_interp_array,
                )
                print(
                    f'[tid={threading.get_ident()}] Saved {len(with_interp_frames)} with_interpreter_frames for {video_path}'
                )
            else:
                print(
                    f'[tid={threading.get_ident()}] No with_interpreter_frames for {video_path}'
                )

            if without_interp_frames:
                without_interp_array = np.array(without_interp_frames)
                np.save(
                    os.path.join(output_dir, 'without_interpreter_frames.npy'),
                    without_interp_array,
                )
                print(
                    f'[tid={threading.get_ident()}] Saved {len(without_interp_frames)} without_interpreter_frames for {video_path}'
                )
            else:
                print(
                    f'[tid={threading.get_ident()}] No without_interpreter_frames for {video_path}'
                )

            return True, video_path
        else:
            return False, video_path

    except Exception as e:
        print(
            f'[tid={threading.get_ident()}] ERROR processing and saving {video_path} → {e!r}'
        )
        return False, video_path


all_with_interpreter_frames = []
all_without_interpreter_frames = []


with ThreadPoolExecutor(max_workers=12, initializer=init_worker) as executor:
    futures_generator = (
        executor.submit(process_video_wrapper, vid_path) for vid_path in video_paths
    )

    for fut in as_completed(futures_generator):
        ok, vid_path = fut.result()
        if ok:
            print(f'[main] Finished processing and saving for: {vid_path}')
        else:
            print(f'[main] Failed or skipped processing for: {vid_path}')

        del fut

[main] 1200 videos to process
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128425607579200] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend


2025-07-09 20:54:17.261753940 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-07-09 20:54:17.261774334 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-07-09 20:54:17.417955993 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1462'. It is not used by any node and should be removed from the model.
2025-07-09 20:54:17.417986933 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1470'. It is not used by any node and should be removed from the model.
2025-07-09 20:54:17.418008025 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1467'. It is not us

load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128423045363264] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128425715037760] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128417668245056] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onn

2025-07-09 20:54:18.718925617 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-07-09 20:54:18.718945592 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-07-09 20:54:18.843310804 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-07-09 20:54:18.843326798 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-07-09 20:54:19.011899477 [W:onnxrun

load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128414774187584] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128413607994944] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/rtmw-dw-l-m_simcc-cocktail14_270e-256x192_20231122.onnx with onnxruntime backend
[tid=128412089816640] wholebody loaded on cuda
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onnxruntime backend
load /home/radumicea/.cache/rtmlib/hub/checkpoints/yolox_tiny_8xb8-300e_humanart-6f3252f9.onnx with onn

2025-07-09 20:54:19.465975798 [W:onnxruntime:, session_state.cc:1280 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-07-09 20:54:19.465992420 [W:onnxruntime:, session_state.cc:1282 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-07-09 20:54:19.598708268 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1462'. It is not used by any node and should be removed from the model.
2025-07-09 20:54:19.598729220 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1470'. It is not used by any node and should be removed from the model.
2025-07-09 20:54:19.598741233 [W:onnxruntime:, graph.cc:4454 CleanUnusedInitializersAndNodeArgs] Removing initializer '1467'. It is not us

[tid=128425607579200] Saved 25 with_interpreter_frames for 2025/05/05/video.mp4
[tid=128425607579200] Saved 15 without_interpreter_frames for 2025/05/05/video.mp4
[main] Finished processing and saving for: 2025/05/05/video.mp4
[tid=128425715037760] Saved 22 with_interpreter_frames for 2025/05/15/video.mp4
[tid=128425715037760] Saved 18 without_interpreter_frames for 2025/05/15/video.mp4
[main] Finished processing and saving for: 2025/05/15/video.mp4
[tid=128423045363264] Saved 25 with_interpreter_frames for 2025/05/22/video.mp4
[tid=128423045363264] Saved 15 without_interpreter_frames for 2025/05/22/video.mp4
[main] Finished processing and saving for: 2025/05/22/video.mp4
[tid=128410688923200] Saved 28 with_interpreter_frames for 2025/05/16/video.mp4
[tid=128410688923200] Saved 12 without_interpreter_frames for 2025/05/16/video.mp4
[main] Finished processing and saving for: 2025/05/16/video.mp4
[tid=128413607994944] Saved 22 with_interpreter_frames for 2025/05/20/video.mp4
[tid=1284136

# Train


In [1]:
def train_net(
    with_interpreter,
    without_interpreter,
    epochs: int = 6,
    batch_size: int = 512,
    lr: float = 1e-3,
    save_path: str = 'interpreter_head.pt',
):  # ────────────────────────────────────────────────────────────────────
    # 1. Imports
    # ────────────────────────────────────────────────────────────────────
    import itertools
    import os
    import time

    import torch
    from PIL import Image
    from torch import nn
    from torch.utils.data import ConcatDataset, DataLoader, Dataset
    from torchvision import models
    from torchvision import transforms as T

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ────────────────────────────────────────────────────────────────────
    # 2.  Determine the native crop size from the first frame
    # ────────────────────────────────────────────────────────────────────
    if not len(with_interpreter) or not len(without_interpreter):
        raise ValueError('No training data!')
    sample_frame = with_interpreter[0]
    h0, w0 = sample_frame.shape[:2]

    # ────────────────────────────────────────────────────────────────────
    # 3.  Dataset
    # ────────────────────────────────────────────────────────────────────
    weights_enum = models.MobileNet_V3_Small_Weights.DEFAULT
    try:
        mean = weights_enum.meta['mean']
        std = weights_enum.meta['std']
    except (KeyError, AttributeError):
        # Standard ImageNet normalisation
        mean = (0.485, 0.456, 0.406)
        std = (0.229, 0.224, 0.225)

    class FrameDataset(Dataset):
        def __init__(self, frames, label, aug=False):
            tfm = []

            if aug:
                tfm += [
                    T.ColorJitter(0.1, 0.1, 0.1, 0.05),
                    T.RandomRotation(5),
                    T.RandomResizedCrop((h0, w0), scale=(0.9, 1.0)),
                ]

            tfm += [T.ToTensor(), T.Normalize(mean, std)]
            self.tfm = T.Compose(tfm)
            self.label = label
            self.data = frames

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            # cv2 loads BGR; convert to RGB
            img = self.data[idx][..., ::-1]
            pil = Image.fromarray(img)
            return self.tfm(pil), self.label

    n_int, n_no = len(with_interpreter), len(without_interpreter)
    # decide which list is minority
    int_is_minority = n_int < n_no
    ds_int = FrameDataset(with_interpreter, 1, aug=int_is_minority)
    ds_no = FrameDataset(without_interpreter, 0, aug=(not int_is_minority))

    full_ds = ConcatDataset([ds_int, ds_no])
    loader = DataLoader(
        full_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=min(4, os.cpu_count() or 1),
        pin_memory=torch.cuda.is_available(),
    )

    # ────────────────────────────────────────────────────────────────────
    # 4.  Model – MobileNet V3 Small (backbone frozen)
    # ────────────────────────────────────────────────────────────────────
    model = models.mobilenet_v3_small(weights=weights_enum)
    in_feats = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_feats, 2)

    for p in itertools.chain(model.features.parameters(), model.avgpool.parameters()):
        p.requires_grad = False
    model = model.to(device)

    # ────────────────────────────────────────────────────────────────────
    # 5.  Loss, optimiser, early stop
    # ────────────────────────────────────────────────────────────────────
    counts = torch.tensor([n_no, n_int], dtype=torch.float, device=device)
    class_wts = counts.sum() / (2 * counts)  # inverse frequency
    criterion = nn.CrossEntropyLoss(weight=class_wts)

    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=lr)

    best_acc, patience = 0.0, 3
    start_time = time.time()

    for epoch in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        # quick accuracy on same loader (enough for early stopping)
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                pred = model(x).argmax(1)
                correct += (pred == y).sum().item()
                total += y.numel()
        acc = correct / total
        print(f'Epoch {epoch + 1:02d}: accuracy={acc:.4f}')

        if acc > best_acc:
            best_acc, patience = acc, 3
        else:
            patience -= 1
        if patience == 0 or best_acc >= 0.995:
            break

    print(
        f'Training finished in {time.time() - start_time:.1f}s '
        f'with accuracy {best_acc:.4f}'
    )

    # ────────────────────────────────────────────────────────────────────
    # 6.  Save weights
    # ────────────────────────────────────────────────────────────────────
    torch.save(model.state_dict(), save_path)
    print(f'Model saved to {os.path.abspath(save_path)}')


In [2]:
import numpy as np
import os


with_interpreter = []
without_interpreter = []


for root, _, files in os.walk('.'):
    for file in files:
        file_path = os.path.join(root, file)

        if file == 'with_interpreter_frames.npy':
            data = np.load(file_path)
            with_interpreter.append(data)

        elif file == 'without_interpreter_frames.npy':
            data = np.load(file_path)
            without_interpreter.append(data)


with_interpreter = np.concatenate(with_interpreter, axis=0)
without_interpreter = np.concatenate(without_interpreter, axis=0)

In [3]:
train_net(with_interpreter, without_interpreter)

Epoch 01: accuracy=0.9882
Epoch 02: accuracy=0.9985
Training finished in 192.6s with accuracy 0.9985
Model saved to /home/radumicea/Projects/University/SLT/scraping/digi/interpreter_head.pt
